# KV cache, measured

Companion notebook to **Lecture 27/28 — Attention and the KV cache**, §5.2. One prefill pass fills Qwen3-0.6B's cache, and the cell below reads it back: tensor shapes, total bytes, and the formula they should match.

- **Model:** `Qwen/Qwen3-0.6B` in BF16, about 1.2 GB. No login needed.
- **Runtime:** the free CPU runtime is enough. One 256-token prefill takes seconds.

**Predict before you run:**

1. The shape of one layer's K tensor for a 256-token prompt. Read `num_hidden_layers`, `num_key_value_heads` and `head_dim` from the model's [`config.json`](https://huggingface.co/Qwen/Qwen3-0.6B/blob/main/config.json).
2. The total KV bytes, from $2 \times \text{layers} \times \text{kv\_heads} \times \text{head\_dim} \times \text{tokens} \times 2$ bytes.
3. The context length at which one conversation's cache outweighs the weights.


In [ ]:
# KV cache: read the shapes, count the bytes, check the formula
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "Qwen/Qwen3-0.6B"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16)
cfg = model.config

# One prefill pass fills the cache for every prompt token.
inputs = tok("The hungry dog ate the bone. " * 37, return_tensors="pt")
inputs["input_ids"] = inputs["input_ids"][:, :256]
inputs["attention_mask"] = inputs["attention_mask"][:, :256]
with torch.no_grad():
    cache = model(**inputs, use_cache=True).past_key_values


def kv_per_layer(cache):
    """(K, V) per layer, across transformers' cache layouts."""
    if hasattr(cache, "layers"):        # DynamicCache, transformers >= 4.56
        return [(layer.keys, layer.values) for layer in cache.layers]
    if hasattr(cache, "key_cache"):     # DynamicCache, older releases
        return list(zip(cache.key_cache, cache.value_cache))
    return list(cache)                  # tuple of (K, V) pairs


kv = kv_per_layer(cache)
n = inputs["input_ids"].shape[1]
k0, v0 = kv[0]

print(f"tokens cached:      {n}")
print(f"layers:             {len(kv)}")
print(f"one layer's K:      {tuple(k0.shape)}   (batch, kv_heads, tokens, head_dim)")
print(f"one layer's V:      {tuple(v0.shape)}   dtype {k0.dtype}")

layer_k = k0.numel() * k0.element_size()
print(f"\none layer's K:      {layer_k:,} bytes   = {' × '.join(map(str, k0.shape))} × {k0.element_size()}")
print(f"K + V, one layer:   {2 * layer_k:,}")
print(f"× {len(kv)} layers:        {2 * layer_k * len(kv):,}")

measured = sum(k.numel() * k.element_size() + v.numel() * v.element_size() for k, v in kv)
formula = 2 * cfg.num_hidden_layers * cfg.num_key_value_heads * cfg.head_dim * n * 2

print(f"\nmeasured KV bytes:  {measured:,}   (summed over every cached tensor)")
print(f"formula (head_dim={cfg.head_dim}): {formula:,}   match: {measured == formula}")

per_token = measured / n
weights = sum(p.numel() * p.element_size() for p in model.parameters())
print(f"\nKV per token:       {per_token / 1024:.0f} KiB")
print(f"weights:            {weights / 1e9:.2f} GB")
print(f"KV = weights at:    {weights / per_token:,.0f} tokens")